# 07. Зведене порівняння всіх моделей

Читає `metrics.json`, `history.csv` і `results.csv` усіх запусків з `hw2_runs/<модель>/<запуск>/`, нічого не перенавчає. Графіки зберігаються в `images/` для звіту.

Швидкість (`ms/кадр`) виміряна на різних GPU: Faster R-CNN — на Kaggle Tesla T4, решта — на RTX 4070 Laptop (колонка `gpu`), тому мс між цими групами напряму не порівнюються. Скори лідерборду — зі сторінки My Submissions на Kaggle (лише для надісланих запусків).

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

RUNS = Path.cwd().parent.parent / "hw2_runs"
IMAGES = Path.cwd().parent / "images"
MODELS = ["faster_rcnn", "retinanet", "yolo", "rtdetr"]
SKIP = ("smoke", "timing")
COLS = ["mAP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "mAP_test_like"]
DEFAULT_GPU = "RTX 4070 Laptop"
LEADERBOARD = {
    ("faster_rcnn", "exp2_giou"): (0.2214, 0.1524),
    ("rtdetr", "baseline"): (0.3178, 0.2221),
    ("rtdetr", "exp1_queries500"): (0.3157, 0.2242),
    ("pipeline", "final"): (0.2101, 0.1671),
    ("pipeline", "final_yolo11m_tiles"): (0.2289, 0.1668),
}
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]


def short_gpu(name):
    return name.replace("NVIDIA GeForce ", "").replace(" GPU", "") if name else DEFAULT_GPU

## 1. Усі запуски

In [ ]:
rows = {}
for path in sorted(RUNS.glob("*/*/metrics.json")):
    model, run = path.parent.parent.name, path.parent.name
    if run.startswith(SKIP) or model not in MODELS:
        continue
    m = json.loads(path.read_text())
    public, private = LEADERBOARD.get((model, run), (None, None))
    rows[(model, run)] = {**{c: m["val"].get(c) for c in COLS}, "ms/кадр": m.get("ms_per_image"),
                          "gpu": short_gpu(m.get("config", {}).get("gpu")), "public_LB": public, "private_LB": private}
runs = pd.DataFrame(rows).T.reindex(MODELS, level=0)
NUMERIC = COLS + ["ms/кадр", "public_LB", "private_LB"]
runs[NUMERIC] = runs[NUMERIC].astype(float)
runs.to_csv(RUNS / "all_runs.csv")
runs.round(4)

## 2. Найкращий запуск кожної моделі (за mAP на val) і фінальні варіанти пайплайну

In [ ]:
best = runs.groupby(level=0, sort=False).apply(lambda g: g.loc[g["mAP"].idxmax()])
best.index = [f"{model} ({runs.loc[model, 'mAP'].idxmax()})" for model in best.index]
for run in ("final", "final_yolo11m_tiles"):
    path = RUNS / "pipeline" / run / "metrics.json"
    if path.exists():
        final = json.loads(path.read_text())
        public, private = LEADERBOARD.get(("pipeline", run), (None, None))
        best.loc[f"pipeline ({final['model']}, {final['variant']})"] = {
            **{c: final["val"].get(c) for c in COLS}, "ms/кадр": None, "gpu": DEFAULT_GPU,
            "public_LB": public, "private_LB": private}
best.round(4)

## 3. Порівняння, яких вимагає умова

RetinaNet (початкова й налаштовані версії) проти налаштованого Faster R-CNN (`exp2_giou`); anchor-free YOLO проти anchor-based моделей за розмірами об'єктів.

In [ ]:
SIZE = ["mAP", "AP50", "AP_small", "AP_medium", "AP_large"]
display(runs.loc[[("faster_rcnn", "exp2_giou"), ("retinanet", "baseline"), ("retinanet", "exp1_anchors"),
                  ("retinanet", "exp2_giou")], SIZE].round(4))
runs.loc[["faster_rcnn", "retinanet", "yolo"], SIZE].round(4)

In [ ]:
sizes = best.loc[[i for i in best.index if not i.startswith("pipeline (yolo11m")], ["AP_small", "AP_medium", "AP_large"]]
ax = sizes.plot.bar(figsize=(10, 4), rot=12, color=COLORS[:3], width=0.75,
                    title="AP за розміром об'єктів (найкращий запуск кожної моделі)")
ax.set_ylabel("AP")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(IMAGES / "ap_by_size.png", dpi=120)
plt.show()

## 4. Val mAP за часом навчання: RT-DETR проти YOLO (той самий бюджет)

Той самий GPU, 960 px, 20 епох. mAP — оцінка Ultralytics на кожному 5-му кадрі val після кожної епохи. Час епохи — різниця колонки `time` у `results.csv`. У YOLO baseline ноутбук засинав і запуск продовжувався з `last.pt`, тож різниці, від'ємні або втричі більші за медіану, замінюємо медіанною тривалістю епохи.

In [ ]:
def epoch_minutes(run_dir):
    results = pd.read_csv(run_dir / "results.csv")
    results.columns = results.columns.str.strip()
    seconds = results["time"].diff().fillna(results["time"])
    median = seconds.median()
    seconds = seconds.where((seconds > 0) & (seconds < 3 * median), median)
    return seconds.cumsum().to_numpy() / 60


fig, ax = plt.subplots(figsize=(10, 4))
timing = {}
for color, (model, run) in zip(COLORS, [("yolo", "baseline"), ("rtdetr", "baseline"), ("rtdetr", "exp1_queries500"),
                                          ("rtdetr", "exp2_giou")]):
    run_dir = RUNS / model / run
    minutes, history = epoch_minutes(run_dir), pd.read_csv(run_dir / "history.csv")
    ax.plot(minutes, history["mAP"], marker="o", markersize=4, linewidth=2, color=color, label=f"{model} {run}")
    timing[f"{model} {run}"] = {"хв навчання": minutes[-1], "найкращий mAP": history["mAP"].max(),
                                "епоха найкращого": int(history.loc[history["mAP"].idxmax(), "epoch"]),
                                "mAP останньої епохи": history["mAP"].iloc[-1]}
ax.set(xlabel="хвилини навчання (RTX 4070 Laptop)", ylabel="mAP@[.5:.95] (кожен 5-й кадр val)",
       title="Val mAP за часом навчання: query-based проти CNN, 20 епох")
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.savefig(IMAGES / "rtdetr_vs_yolo.png", dpi=120)
plt.show()
pd.DataFrame(timing).T.round(4)

## 5. Val mAP за епохами: Faster R-CNN і RetinaNet

Оцінка `hw2_eval` на кожному 5-му кадрі val. Faster R-CNN — 12 епох (Tesla T4), RetinaNet — 6 епох (RTX 4070 Laptop).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
styles = {"faster_rcnn": "-", "retinanet": "--"}
for model in ("faster_rcnn", "retinanet"):
    for color, path in zip(COLORS, sorted((RUNS / model).glob("*/history.csv"))):
        if path.parent.name.startswith(SKIP):
            continue
        h = pd.read_csv(path)
        ax.plot(h["epoch"], h["mAP"], styles[model], marker="o", markersize=4, linewidth=2, color=color,
                label=f"{model} {path.parent.name}")
ax.set(xlabel="епоха", ylabel="mAP@[.5:.95] (кожен 5-й кадр val)", title="Two-stage vs one-stage anchor-based",
       xticks=range(1, 13))
ax.grid(alpha=0.3)
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.savefig(IMAGES / "frcnn_vs_retinanet.png", dpi=120)
plt.show()

## 6. Пайплайн: внесок інференсу (A, YOLO11s) і моделі (B, YOLO11m)

In [ ]:
path = RUNS / "pipeline" / "comparison.csv"
pd.read_csv(path, header=[0, 1], index_col=0).round(4) if path.exists() else "ще немає"

## Джерела

- pandas, Matplotlib — таблиці й графіки.
- Метрики — спільний оцінювач `hw2_eval.py` (faster-coco-eval, COCO detection evaluation); для кривих YOLO / RT-DETR — вбудована валідація Ultralytics (`results.csv`).
- Скори лідерборду — сторінка My Submissions змагання на Kaggle.

**Наші рішення:** найкращий запуск за mAP на повному val; заміна аномальних тривалостей епох (сон ноутбука, перезапуск з `last.pt`) медіанною тривалістю.